In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType,
    ArrayType, DoubleType, IntegerType
)

spark = SparkSession.builder.appName("NestedJSONFlattenExample").getOrCreate()

event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_ts", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("customer", StructType([
        StructField("customer_id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("segment", StringType(), True),
        StructField("contact", StructType([
            StructField("email", StringType(), True),
            StructField("phones", ArrayType(StructType([
                StructField("type", StringType(), True),
                StructField("number", StringType(), True),
            ])), True),
        ])),
        StructField("addresses", ArrayType(StructType([
            StructField("address_id", StringType(), True),
            StructField("type", StringType(), True),
            StructField("line1", StringType(), True),
            StructField("city", StringType(), True),
            StructField("state", StringType(), True),
            StructField("pin_code", StringType(), True),
        ])), True),
    ])),
    StructField("order", StructType([
        StructField("order_id", StringType(), True),
        StructField("order_ts", StringType(), True),
        StructField("currency", StringType(), True),
        StructField("total_amount", DoubleType(), True),
        StructField("status", StringType(), True),
        StructField("items", ArrayType(StructType([
            StructField("item_id", StringType(), True),
            StructField("sku", StringType(), True),
            StructField("name", StringType(), True),
            StructField("qty", IntegerType(), True),
            StructField("unit_price", DoubleType(), True),
            StructField("tags", ArrayType(StringType(), True), True),
        ])), True),
        StructField("payment", StructType([
            StructField("method", StringType(), True),
            StructField("card_type", StringType(), True),
            StructField("masked_card_number", StringType(), True),
            StructField("gateway", StringType(), True),
            StructField("transaction_id", StringType(), True),
        ])),
    ])),
    StructField("device", StructType([
        StructField("device_id", StringType(), True),
        StructField("type", StringType(), True),
        StructField("os", StringType(), True),
        StructField("app_version", StringType(), True),
    ])),
    StructField("metadata", StructType([
        StructField("ingest_ts", StringType(), True),
        StructField("ingest_region", StringType(), True),
        StructField("retry_count", IntegerType(), True),
    ])),
])

# Simulate JSON ingestion (list with a single dict here)
json_data = [  # paste the JSON dict here as Python dict
    {
        "event_id": "EVT-1001",
        "event_ts": "2026-07-29T10:15:30Z",
        "source_system": "web",
        "customer": {
            "customer_id": "CUST-001",
            "name": "Alice",
            "segment": "premium",
            "contact": {
                "email": "alice@example.com",
                "phones": [
                    {"type": "mobile", "number": "+91-9876543210"},
                    {"type": "home", "number": "+91-8044123456"}
                ]
            },
            "addresses": [
                {
                    "address_id": "ADDR-100",
                    "type": "home",
                    "line1": "10, MG Road",
                    "city": "Bengaluru",
                    "state": "Karnataka",
                    "pin_code": "560001"
                },
                {
                    "address_id": "ADDR-101",
                    "type": "office",
                    "line1": "3rd Floor, Tech Park",
                    "city": "Bengaluru",
                    "state": "Karnataka",
                    "pin_code": "560103"
                }
            ]
        },
        "order": {
            "order_id": "ORD-9001",
            "order_ts": "2026-07-29T10:10:00Z",
            "currency": "INR",
            "total_amount": 1500.75,
            "status": "confirmed",
            "items": [
                {
                    "item_id": "ITEM-001",
                    "sku": "SKU-100",
                    "name": "Wireless Mouse",
                    "qty": 2,
                    "unit_price": 500.00,
                    "tags": ["electronics", "accessory"]
                },
                {
                    "item_id": "ITEM-002",
                    "sku": "SKU-200",
                    "name": "Keyboard",
                    "qty": 1,
                    "unit_price": 500.75,
                    "tags": ["electronics"]
                }
            ],
            "payment": {
                "method": "card",
                "card_type": "credit",
                "masked_card_number": "XXXX-XXXX-XXXX-1234",
                "gateway": "Razorpay",
                "transaction_id": "TXN-777777"
            }
        },
        "device": {
            "device_id": "DEV-ABC",
            "type": "mobile",
            "os": "Android",
            "app_version": "3.2.1"
        },
        "metadata": {
            "ingest_ts": "2026-07-29T10:16:00Z",
            "ingest_region": "ap-south-1",
            "retry_count": 0
        }
    }
]

bronze_df = spark.createDataFrame(json_data, schema=event_schema)
bronze_df.printSchema()
bronze_df.show(truncate=False)

In [0]:
from pyspark.sql.functions import col, explode_outer
from pyspark.sql import DataFrame

In [0]:
def flatten_events(df: DataFrame) -> DataFrame:
    return df.select(
        col("event_id"),
        col("event_ts"),
        col("source_system"),
        col("metadata.ingest_ts").alias("ingest_ts"),
        col("metadata.ingest_region").alias("ingest_region"),
        col("metadata.retry_count").alias("retry_count"),
        col("device.device_id").alias("device_id"),
        col("device.type").alias("device_type"),
        col("device.os").alias("device_os"),
        col("device.app_version").alias("device_app_version"),
        col("customer.customer_id").alias("customer_id"),
        col("order.order_id").alias("order_id"),
    )

events_silver_df = flatten_events(bronze_df)
events_silver_df.show(truncate=False)

In [0]:
def flatten_customers(df: DataFrame) -> DataFrame:
    return df.select(
        col("customer.customer_id").alias("customer_id"),
        col("customer.name").alias("customer_name"),
        col("customer.segment").alias("segment"),
        col("customer.contact.email").alias("email")
    ).dropDuplicates(["customer_id"])  # dimension-style

customers_silver_df = flatten_customers(bronze_df)
customers_silver_df.show(truncate=False)

In [0]:
def flatten_customer_phones(df: DataFrame) -> DataFrame:
    exploded = df.select(
        col("event_id"),
        col("customer.customer_id").alias("customer_id"),
        explode_outer(col("customer.contact.phones")).alias("phone")
    )
    return exploded.select(
        "event_id",
        "customer_id",
        col("phone.type").alias("phone_type"),
        col("phone.number").alias("phone_number")
    )

customer_phones_df = flatten_customer_phones(bronze_df)
customer_phones_df.show(truncate=False)

In [0]:
def flatten_customer_addresses(df: DataFrame) -> DataFrame:
    exploded = df.select(
        col("event_id"),
        col("customer.customer_id").alias("customer_id"),
        explode_outer(col("customer.addresses")).alias("addr")
    )
    return exploded.select(
        "event_id",
        "customer_id",
        col("addr.address_id").alias("address_id"),
        col("addr.type").alias("address_type"),
        col("addr.line1").alias("line1"),
        col("addr.city").alias("city"),
        col("addr.state").alias("state"),
        col("addr.pin_code").alias("pin_code")
    )

customer_addresses_df = flatten_customer_addresses(bronze_df)
customer_addresses_df.show(truncate=False)

In [0]:
def flatten_customer_addresses(df: DataFrame) -> DataFrame:
    exploded = df.select(
        col("event_id"),
        col("customer.customer_id").alias("customer_id"),
        explode_outer(col("customer.addresses")).alias("addr")
    )
    return exploded.select(
        "event_id",
        "customer_id",
        col("addr.address_id").alias("address_id"),
        col("addr.type").alias("address_type"),
        col("addr.line1").alias("line1"),
        col("addr.city").alias("city"),
        col("addr.state").alias("state"),
        col("addr.pin_code").alias("pin_code")
    )

customer_addresses_df = flatten_customer_addresses(bronze_df)
customer_addresses_df.show(truncate=False)

In [0]:
def flatten_orders(df: DataFrame) -> DataFrame:
    return df.select(
        col("order.order_id").alias("order_id"),
        col("order.order_ts").alias("order_ts"),
        col("order.currency").alias("currency"),
        col("order.total_amount").alias("total_amount"),
        col("order.status").alias("status"),
        col("order.payment.method").alias("payment_method"),
        col("order.payment.card_type").alias("card_type"),
        col("order.payment.masked_card_number").alias("masked_card_number"),
        col("order.payment.gateway").alias("payment_gateway"),
        col("order.payment.transaction_id").alias("transaction_id"),
        col("event_id"),
        col("customer.customer_id").alias("customer_id")
    )

orders_df = flatten_orders(bronze_df)
orders_df.show(truncate=False)

In [0]:
def flatten_order_items(df: DataFrame) -> DataFrame:
    exploded = df.select(
        col("event_id"),
        col("order.order_id").alias("order_id"),
        explode_outer(col("order.items")).alias("item")
    )
    return exploded.select(
        "event_id",
        "order_id",
        col("item.item_id").alias("item_id"),
        col("item.sku").alias("sku"),
        col("item.name").alias("item_name"),
        col("item.qty").alias("qty"),
        col("item.unit_price").alias("unit_price")
    )

order_items_df = flatten_order_items(bronze_df)
order_items_df.show(truncate=False)

In [0]:
def flatten_order_item_tags(df: DataFrame) -> DataFrame:
    # First explode items, then tags inside each item[21][12]
    exploded_items = df.select(
        col("event_id"),
        col("order.order_id").alias("order_id"),
        explode_outer(col("order.items")).alias("item")
    )

    exploded_tags = exploded_items.select(
        "event_id",
        "order_id",
        col("item.item_id").alias("item_id"),
        explode_outer(col("item.tags")).alias("tag")
    )

    return exploded_tags.select(
        "event_id",
        "order_id",
        "item_id",
        col("tag").alias("tag")
    )

order_item_tags_df = flatten_order_item_tags(bronze_df)
order_item_tags_df.show(truncate=False)